## 1. EDA

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt

In [ ]:
# This seems to be a "synthesized" dataset from various original tables from https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
# retrieved from https://www.kaggle.com/code/andresionek/predicting-customer-satisfaction/input
orders = pd.read_csv('./input/olist_public_dataset_v2.csv')

# converting to datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders.order_purchase_timestamp)
orders['order_aproved_at'] = pd.to_datetime(orders.order_aproved_at).dt.date  
orders['order_estimated_delivery_date'] = pd.to_datetime(orders.order_estimated_delivery_date).dt.date  
orders['order_delivered_customer_date'] = pd.to_datetime(orders.order_delivered_customer_date).dt.date  

# get translations for category names
translation = pd.read_csv('./input/product_category_name_translation.csv')
orders = orders.merge(translation, on='product_category_name').drop('product_category_name', axis=1)

orders.head(3)

,order_id,order_status,order_products_value,order_freight_value,order_items_qty,order_sellers_qty,order_purchase_timestamp,order_aproved_at,order_estimated_delivery_date,order_delivered_customer_date,...,product_description_lenght,product_photos_qty,product_id,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,product_category_name_english
0,b95df3cef5297e79ef709ba256518f6f,delivered,349.90,13.84,1,1,2017-01-31 17:19:01,2017-02-01,2017-03-15,2017-02-06,...,625,1,6cdd53843498f92890544667809f1595,b95df3cef5297e79ef709ba256518f6f,5,NaN,NaN,2017-02-07 00:00:00.000000,2017-02-09 02:37:37+00:00,health_beauty
1,e98077a0d199a25a40eab3b14cc230d4,delivered,39.99,15.23,1,2,2018-04-17 13:25:36,2018-04-18,2018-05-10,2018-04-27,...,405,2,190d9562bfbe9d3ed876c2ac6f2f5894,e98077a0d199a25a40eab3b14cc230d4,5,NaN,NaN,2018-04-28 00:00:00.000000,2018-04-29 21:07:53+00:00,health_beauty
2,8a723730400b508cbf47fbef4a76ec8e,delivered,60.00,20.91,1,1,2018-02-18 12:41:01,2018-02-18,2018-03-14,2018-03-03,...,1665,1,5858f45c20fde7d7e49af37a2166635a,8a723730400b508cbf47fbef4a76ec8e,5,NaN,muito bom cabelo fica lisinho,2018-03-04 00:00:00.000000,2018-03-07 02:53:50+00:00,health_beauty


In [38]:
# check all "NaT" in the dataset
time_cols = ['order_purchase_timestamp', 'order_aproved_at', 'order_estimated_delivery_date', 'order_delivered_customer_date']
for col in time_cols:
    print(col)
    print(f"Number of NaT: {orders[col].isnull().sum()}")
    print(orders[orders[col].isnull()].index)
    print()

order_purchase_timestamp
Number of NaT: 0
Int64Index([], dtype='int64')

order_aproved_at
Number of NaT: 18
Int64Index([ 1690,  1792,  8819, 18814, 20149, 29611, 30217, 32192, 37264,
            37639, 52907, 56136, 73317, 80639, 82948, 83464, 84083, 93241],
           dtype='int64')

order_estimated_delivery_date
Number of NaT: 0
Int64Index([], dtype='int64')

order_delivered_customer_date
Number of NaT: 2405
Int64Index([   65,    85,    97,   170,   216,   234,   298,   319,   346,
              396,
            ...
            99817, 99852, 99863, 99886, 99888, 99891, 99914, 99957, 99982,
            99997],
           dtype='int64', length=2405)



In [40]:
# for NaT in "order_aproved_at", use the date from "order_purchase_timestamp"
orders.loc[orders['order_aproved_at'].isnull(), 'order_aproved_at'] = orders.loc[orders['order_aproved_at'].isnull(), 'order_purchase_timestamp']
orders['order_aproved_at'] = pd.to_datetime(orders['order_aproved_at'])

/tmp/ipykernel_10603/1624312865.py:3: FutureWarning: Comparison of Timestamp with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable. Use 'ts == pd.Timestamp(date)' or 'ts.date() == date' instead.
  orders['order_aproved_at'] = pd.to_datetime(orders['order_aproved_at'])
/tmp/ipykernel_10603/1624312865.py:3: FutureWarning: Comparison of Timestamp with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable. Use 'ts == pd.Timestamp(date)' or 'ts.date() == date' instead.
  orders['order_aproved_at'] = pd.to_datetime(orders['order_aproved_at'])


In [42]:
print(f"Number of NaT: {orders['order_aproved_at'].isnull().sum()}")

Number of NaT: 0


## 2. Data Processing

### a. Drop Cols

In [43]:
orders = orders[['order_status', 'order_products_value',
                 'order_freight_value', 'order_items_qty', 'order_sellers_qty',
                 'order_purchase_timestamp', 'order_aproved_at', 'order_estimated_delivery_date', 
                 'order_delivered_customer_date', 'customer_state', 
                 'product_category_name_english', 'product_name_lenght', 'product_description_lenght', 
                 'product_photos_qty', 'review_score']]

### b. Spliting Data

In [44]:
# We keep the same proportion of classes
orders['review_score'].value_counts() / len(orders['review_score'])

5    0.56693
4    0.19247
1    0.11905
3    0.08743
2    0.03412
Name: review_score, dtype: float64

In [45]:
from sklearn.model_selection import StratifiedShuffleSplit

# Stratified Split
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(orders, orders['review_score']):
    strat_train_set = orders.loc[train_index]
    strat_test_set = orders.loc[test_index]

In [46]:
strat_train_set['review_score'].value_counts() / len(strat_train_set['review_score'])

5    0.566925
4    0.192475
1    0.119050
3    0.087425
2    0.034125
Name: review_score, dtype: float64

### c. Separate Labels From Features

In [47]:
orders_features = strat_train_set.drop('review_score', axis=1)
orders_labels = strat_train_set['review_score'].copy()

### d. Feature Engineering

From the existing features, we can see that there are weak linear correlations between the features and the target variable. Thus we should create new features that can help the model to learn better.

In [48]:
corr_matrix = strat_train_set.corr()
corr_matrix['review_score'].sort_values(ascending=False)

/tmp/ipykernel_10603/81064377.py:1: FutureWarning: The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  corr_matrix = strat_train_set.corr()


review_score                  1.000000
product_photos_qty            0.017596
product_description_lenght    0.014032
product_name_lenght          -0.007707
order_products_value         -0.020501
order_freight_value          -0.071941
order_items_qty              -0.073601
order_sellers_qty            -0.137920
Name: review_score, dtype: float64

In [49]:
from sklearn.base import BaseEstimator, TransformerMixin
from workalendar.america import Brazil


cal = Brazil()


class AttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = X.copy()

        # Calculate the estimated delivery time and actual delivery time in working days.
        # This would allow us to exclude hollidays that could influence delivery times.
        # If the order_delivered_customer_date is null, it returns 0.
        df["wd_estimated_delivery_time"] = df.apply(
            lambda x: cal.get_working_days_delta(
                x.order_aproved_at, x.order_estimated_delivery_date
            ),
            axis=1,
        )
        df["wd_actual_delivery_time"] = df.apply(
            lambda x: cal.get_working_days_delta(
                x.order_aproved_at, x.order_delivered_customer_date
            ),
            axis=1,
        )

        # Calculate the time between the actual and estimated delivery date. If negative was delivered early, if positive was delivered late.
        df["wd_delivery_time_delta"] = (
            df.wd_actual_delivery_time - df.wd_estimated_delivery_time
        )

        # Calculate the time between the actual and estimated delivery date. If negative was delivered early, if positive was delivered late.
        df["is_late"] = (
            df.order_delivered_customer_date > df.order_estimated_delivery_date
        )

        # Calculate the average product value.
        df["average_product_value"] = df.order_products_value / df.order_items_qty

        # Calculate the total order value
        df["total_order_value"] = df.order_products_value + df.order_freight_value

        # Calculate the order freight ratio.
        df["order_freight_ratio"] = df.order_freight_value / df.order_products_value

        # Calculate the order freight ratio.
        df["purchase_dayofweek"] = df.order_purchase_timestamp.dt.dayofweek

        # With that we can remove the timestamps from the dataset
        cols2drop = [
            "order_purchase_timestamp",
            "order_aproved_at",
            "order_estimated_delivery_date",
            "order_delivered_customer_date",
        ]
        df.drop(cols2drop, axis=1, inplace=True)

        return df

In [50]:
attr_adder = AttributesAdder()
feat_eng = attr_adder.transform(strat_train_set)
feat_eng.head(3)

/home/joshuale/anaconda3/envs/databricks-mlops/lib/python3.11/site-packages/workalendar/core.py:862: FutureWarning: Comparison of NaT with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable.
  if start > end:
/home/joshuale/anaconda3/envs/databricks-mlops/lib/python3.11/site-packages/workalendar/core.py:872: FutureWarning: Comparison of NaT with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable.
  while start < end:


,order_status,order_products_value,order_freight_value,order_items_qty,order_sellers_qty,customer_state,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,review_score,wd_estimated_delivery_time,wd_actual_delivery_time,wd_delivery_time_delta,is_late,average_product_value,total_order_value,order_freight_ratio,purchase_dayofweek
41966,delivered,45.9,8.72,1,1,SP,sports_leisure,39,772,1,5,8,5,-3,False,45.9,54.62,0.189978,0
62325,delivered,45.0,17.25,1,1,BA,watches_gifts,56,660,3,5,12,14,2,True,45.0,62.25,0.383333,3
7720,delivered,89.9,15.70,1,1,SP,health_beauty,49,819,3,5,22,2,-20,False,89.9,105.60,0.174638,3


In [51]:
corr_matrix = feat_eng.corr()
corr_matrix['review_score'].sort_values(ascending=False)

/tmp/ipykernel_10603/1186622368.py:1: FutureWarning: The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  corr_matrix = feat_eng.corr()


review_score                  1.000000
product_photos_qty            0.017596
product_description_lenght    0.014032
average_product_value        -0.003409
purchase_dayofweek           -0.006675
product_name_lenght          -0.007707
order_products_value         -0.020501
order_freight_ratio          -0.023914
total_order_value            -0.026802
wd_estimated_delivery_time   -0.053587
order_freight_value          -0.071941
order_items_qty              -0.073601
order_sellers_qty            -0.137920
wd_delivery_time_delta       -0.171696
wd_actual_delivery_time      -0.238200
is_late                      -0.338739
Name: review_score, dtype: float64

### e. Dealing with Categorical and Numerical Attributes

In [52]:
cat_attribs = ['order_status', 'customer_state', 'product_category_name_english']
num_attribs = orders_features.drop(cat_attribs, axis=1).columns

In [53]:
class DataFrameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, attribute_names):
        self.attribute_names = attribute_names
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        return X[self.attribute_names]

In [54]:
# Numerical Attributes¶
# Creating pipelines to handle unseen data

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# for now we wont work with categorical data. Planning to add it on next releases
num_pipeline = Pipeline([('selector', DataFrameSelector(num_attribs)),
                         ('attribs_adder', AttributesAdder()),
                         ('std_scaller', StandardScaler())
                        ])

In [55]:
orders_features_prepared = num_pipeline.fit_transform(orders_features)
orders_features_prepared

/home/joshuale/anaconda3/envs/databricks-mlops/lib/python3.11/site-packages/workalendar/core.py:862: FutureWarning: Comparison of NaT with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable.
  if start > end:
/home/joshuale/anaconda3/envs/databricks-mlops/lib/python3.11/site-packages/workalendar/core.py:872: FutureWarning: Comparison of NaT with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable.
  while start < end:


array([[-0.42864476, -0.64029152, -0.21753729, ..., -0.47356485,
        -0.37769827, -1.40790805],
       [-0.43329345, -0.221773  , -0.21753729, ..., -0.43594081,
         0.27782067,  0.11674387],
       [-0.20137565, -0.29782267, -0.21753729, ..., -0.22217906,
        -0.42970352,  0.11674387],
       ...,
       [-0.29476259,  0.79876456, -0.21753729, ..., -0.2011234 ,
         0.77436568,  1.13317848],
       [-0.14972358, -0.07261751, -0.21753729, ..., -0.15023479,
        -0.33320183, -1.40790805],
       [-0.40798393, -0.1864467 , -0.21753729, ..., -0.40822822,
         0.19912287,  1.64139579]])

In [56]:
some_data = orders_features.iloc[:8]
some_labels = orders_labels.iloc[:8]
some_data_prepared = num_pipeline.transform(some_data)

In [57]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error



forest_reg = RandomForestRegressor()
forest_reg.fit(orders_features_prepared, orders_labels)

predictions = forest_reg.predict(orders_features_prepared)
forest_mse = mean_squared_error(orders_labels, predictions)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

0.45651917310526874

In [58]:
print('Predicted: {} \n Labels: {}'.format(list(forest_reg.predict(some_data_prepared)), list(some_labels.values)))

Predicted: [4.75, 4.17, 4.83, 4.14, 3.4, 4.9, 1.36, 1.7866666666666666] 
 Labels: [5, 5, 5, 4, 4, 5, 1, 1]
